##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [1]:
import os
import cv2
import random
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from collections import deque
import matplotlib.pyplot as plt

# Fix random seeds for reproducibility
seed_constant = 42
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

# Constants
image_height, image_width = 64, 64
max_images_per_class = 2000       # number of frames per class
dataset_dir = "UCF11_updated_mp4v2"
selected_classes = ["horse_riding", "trampoline_jumping", "walking"]
model_output_size = len(selected_classes)

In [2]:
# Download the UCF11 dataset (approx 2.5 GB)
!wget -nc https://www.crcv.ucf.edu/data/UCF11_updated_mp4v2.rar
!unrar x UCF11_updated_mp4v2.rar -inul -y
print("Dataset extracted into folder:", dataset_dir)

'wget' is not recognized as an internal or external command,
operable program or batch file.


Dataset extracted into folder: UCF11_updated_mp4v2


'unrar' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
def frames_extraction(video_path):
    """Extract, resize, and normalize all frames from a video."""
    frames_list = []
    cap = cv2.VideoCapture(video_path)
    while True:
        success, frame = cap.read()
        if not success:
            break
        resized = cv2.resize(frame, (image_height, image_width))
        normalized = resized / 255.0
        frames_list.append(normalized)
    cap.release()
    return frames_list

def create_dataset():
    """For each selected class, sample max_images_per_class random frames."""
    features, labels = [], []
    for class_idx, class_name in enumerate(selected_classes):
        print(f"Processing class: {class_name}")
        class_path = os.path.join(dataset_dir, class_name)
        video_files = os.listdir(class_path)
        temp_frames = []
        for vfile in video_files:
            vpath = os.path.join(class_path, vfile)
            temp_frames.extend(frames_extraction(vpath))
        # Randomly sample frames from this class
        sampled = random.sample(temp_frames, min(max_images_per_class, len(temp_frames)))
        features.extend(sampled)
        labels.extend([class_idx] * len(sampled))
        print(f"  -> Added {len(sampled)} frames")
    return np.asarray(features), np.array(labels)

In [4]:
features, labels = create_dataset()
print("Features shape:", features.shape)
print("Labels shape:", labels.shape)

# One‑hot encode labels
one_hot_labels = to_categorical(labels, num_classes=model_output_size)

Processing class: horse_riding


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'UCF11_updated_mp4v2\\horse_riding'

In [ ]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    features, one_hot_labels, test_size=0.2, shuffle=True, random_state=seed_constant
)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

# Build CNN model
def create_model():
    model = Sequential([
        Conv2D(64, (3,3), activation='relu', input_shape=(image_height, image_width, 3)),
        Conv2D(64, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dense(model_output_size, activation='softmax')
    ])
    model.summary()
    return model

model = create_model()
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Early stopping callback
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
student_name = "YourName"   # <---- CHANGE THIS TO YOUR ACTUAL NAME
save_path = f"{student_name}_ucf11_model.h5"
model.save(save_path)
print(f"Model saved as {save_path}")

In [ ]:
!pip install yt-dlp -q
import yt_dlp

def download_youtube_video(url, output_name):
    ydl_opts = {
        'outtmpl': f'{output_name}.mp4',
        'format': 'mp4',
        'quiet': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

# === REPLACE THESE URLs WITH REAL ONES ===
video_urls = {
    "horse_riding": "https://youtu.be/ohr_Vex4m4c?si=qlP3IK8zBwpK4VfH",
    "trampoline_jumping": "https://youtu.be/LMzBmhtTFlQ?si=iLf0JKE3cVHktPP_",
    "walking": "https://youtu.be/84lYjtCfIvY?si=eabDwo0PvWfv6Tbl"
}

for action, url in video_urls.items():
    print(f"Downloading {action}.mp4 ...")
    download_youtube_video(url, action)
    print(f"Finished: {action}.mp4")

In [ ]:
def predict_on_video(video_path, window_size=15):
    """
    Reads a video, predicts action frame‑by‑frame using a moving average,
    and saves an output video with the predicted label overlaid.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video {video_path}")
        return
    
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 24
    
    out_path = f"output_{os.path.basename(video_path)}"
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
    
    prob_deque = deque(maxlen=window_size)
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Preprocess frame
        resized = cv2.resize(frame, (image_height, image_width))
        normalized = resized / 255.0
        input_tensor = np.expand_dims(normalized, axis=0)
        pred_probs = model.predict(input_tensor, verbose=0)[0]
        prob_deque.append(pred_probs)
        
        if len(prob_deque) == window_size:
            avg_probs = np.mean(prob_deque, axis=0)
            pred_class_idx = np.argmax(avg_probs)
            pred_class_name = selected_classes[pred_class_idx]
            # Put text on the frame
            cv2.putText(frame, pred_class_name, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                        1, (0, 255, 0), 2)
        
        out.write(frame)
    
    cap.release()
    out.release()
    print(f"Saved output video: {out_path}")

# Run prediction on the three downloaded videos
for action in video_urls.keys():
    video_file = f"{action}.mp4"
    if os.path.exists(video_file):
        print(f"\n--- Predicting on {video_file} ---")
        predict_on_video(video_file, window_size=20)
    else:
        print(f"Video {video_file} not found. Please check the download step.")

In [ ]:
def average_prediction_over_video(video_path, num_frames=50):
    """Sample num_frames evenly from the video and average predictions."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    skip = max(1, total_frames // num_frames)
    prob_sum = np.zeros(model_output_size)
    count = 0
    for i in range(0, total_frames, skip):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            continue
        resized = cv2.resize(frame, (image_height, image_width)) / 255.0
        probs = model.predict(np.expand_dims(resized, axis=0), verbose=0)[0]
        prob_sum += probs
        count += 1
    cap.release()
    avg_probs = prob_sum / count
    pred_idx = np.argmax(avg_probs)
    print(f"\nResults for {video_path}:")
    for i, cls in enumerate(selected_classes):
        print(f"  {cls}: {avg_probs[i]:.4f}")
    print(f"  -> Predicted action: {selected_classes[pred_idx]}")

# Example
for action in video_urls.keys():
    average_prediction_over_video(f"{action}.mp4")